# Supervised Fraud Detection and Risk Scoring

This notebook trains and evaluates **supervised fraud detection models**, including a **Random Forest baseline** and a **LightGBM** model as the primary approach.

The focus is on **ranking transactions by fraud risk**, with **PR-AUC as the primary evaluation metric** due to extreme class imbalance, and **ROC-AUC used for additional context.**

Model performance is further analysed at a **fixed 0.5% alert rate** to examine performance among the highest-risk transactions.

The resulting supervised risk scores are then **prepared and saved** for use in a subsequent **hybrid fraud prioritisation** stage that incorporates behavioural anomaly signals.

## 1.Setup Environment and Load Processed Data

In [ ]:
#mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#import required libraries

import numpy as np
import pandas as pd
import joblib
import os
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.metrics import roc_curve

In [ ]:
#load processed datasets

data_folder='/content/drive/MyDrive/hybrid-fraud-risk-prioritisation/data'

X_train=joblib.load(f'{data_folder}/X_train.pkl')
y_train = joblib.load(f'{data_folder}/y_train.pkl')
X_test = joblib.load(f'{data_folder}/X_test.pkl')
y_test = joblib.load(f'{data_folder}/y_test.pkl')

## 2.Supervised Models

### 2.1.Random Forest

In [ ]:
#train and evaluate a Random Forest classifier

#initialise the model
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

#fit the model on the train data
rf.fit(X_train, y_train)

#generate fraud probability scores on the test set
y_test_proba_rf = rf.predict_proba(X_test)[:, 1]

#print ranking performance using PR-AUC and ROC-AUC
print('RF PR-AUC:', average_precision_score(y_test, y_test_proba_rf))
print('RF ROC-AUC:', roc_auc_score(y_test, y_test_proba_rf))

### 2.2. LightGBM

In [ ]:
#train and evaluate a class-weighted LightGBM model

#count non-fraud and fraud instances in the train set
num_neg=(y_train==0).sum()
num_pos=(y_train==1).sum()

#handle class imbalance using class weighting (used instead of SMOTE)
scale_pos_weight=num_neg/num_pos

#initialise the model
lgbm=LGBMClassifier(
    objective='binary',
    boosting_type='gbdt',
    device='gpu',
    scale_pos_weight=scale_pos_weight,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

#fit the model on train data
lgbm.fit(X_train, y_train)

#generate fraud probability of class = 1 for test data
y_test_proba=lgbm.predict_proba(X_test)[:, 1]

#calculate and print ranking performance using PR-AUC and ROC-AUC
pr_auc = average_precision_score(y_test, y_test_proba)
roc_auc = roc_auc_score(y_test, y_test_proba)

print(f'PR-AUC:  {pr_auc:.6f}')
print(f'ROC-AUC: {roc_auc:.6f}')

**Note:**

With a PR-AUC of approximately **0.0089** and an ROC-AUC of **0.6030**,  LightGBM model performs better than random ranking, but shows **weak fraud concentration and limited separability**, indicating that it struggles to clearly prioritise and distinguish fraud cases on its own.

In [ ]:
#evaluate model performance when only the top 0.5% highest-risk transactions are flagged

#define alert rate and derive probability threshold based on top-risk percentile
alert_rate=0.5
threshold=np.percentile(y_test_proba, 100-alert_rate)

#flag transactions above the risk threshold as fraud alerts
y_pred=(y_test_proba>=threshold).astype(int)

#compute confusion matrix at the selected alert rate
cm=confusion_matrix(y_test, y_pred)
TN, FP, FN, TP=cm.ravel()

#visualise confusion matrix
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt=',',
    cmap='Blues',
    xticklabels=['Not Fraud', 'Fraud'],
    yticklabels=['Not Fraud', 'Fraud']
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (Top 0.5% Risk Threshold)')
plt.tight_layout()
plt.show()

#compute precision and recall at 0.5% alert rate
precision_val=TP/(TP+FP)
recall_val=TP/(TP+FN)

print(f"Threshold used: {threshold:.6f}")
print(f"Precision at 0.5% alert rate: {precision_val:.4%}")
print(f"Recall at 0.5% alert rate: {recall_val:.4%}")

In [ ]:
#compute and plot ROC curve to assess overall separability
fpr, tpr, thresholds = roc_curve(y_test, y_test_proba)
roc_auc=auc(fpr, tpr)

plt.figure(figsize=(10,7))
plt.plot(fpr, tpr, label=f'LGBM ROC AUC: {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (LightGBM)')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#compute and plot precision-recall curve to assess fraud concentration at high-risk ranks
precision, recall, _ = precision_recall_curve(y_test, y_test_proba)
pr_auc = average_precision_score(y_test, y_test_proba)

plt.figure(figsize=(10,7))
plt.plot(recall, precision, label=f'LightGBM PR-AUC: {pr_auc:.4f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (LightGBM)')
plt.legend()
plt.grid(True)
plt.show()

**Note:**

The Precision-Recall curve shows that precision drops quickly as recall increases, indicating that only a small number of fraud cases are clearly identifiable using transaction-level features alone. While the model correctly ranks a few high-risk frauds at the top, precision rapidly approaches the baseline fraud rate as more transactions are flagged. This behaviour is reflected in the low PR-AUC score and motivates the use of behavioural anomaly features in the next stage.

## 3.Prepare and Save Supervised Outputs for Hybrid Fraud Prioritisation


In [ ]:
#load the saved stratified 5M-row sample for result alignment

df=pd.read_parquet(f'{data_folder}/sample_5M_df.parquet')

In [ ]:
#add transaction identifiers to the test-set results
supervised_results = df.loc[X_test.index, ['nameOrig', 'day']].copy()

#attach supervised fraud probability scores and true labels
supervised_results['supervised_score'] = y_test_proba
supervised_results['isFraud'] = y_test.values

In [ ]:
supervised_results.info()

In [ ]:
#save supervised scores for downstream hybrid fraud prioritisation

output_dir='/content/drive/MyDrive/hybrid-fraud-risk-prioritisation/outputs'
os.makedirs(output_dir, exist_ok=True)

supervised_results.to_parquet(
    f'{output_dir}/supervised_scores.parquet', index=False
)